# MMASH Bridge Test: Sleep Efficiency → Stress

This notebook tests whether the MMASH dataset can solve the missing bridge:

sleep efficiency → stress

MMASH is useful because it contains sleep variables, activity logs, stress, anxiety, and emotion measures.

Main target path:

Latency Efficiency → Daily_stress

If this works, it can support the connection:

alcohol/exercise → sleep efficiency → stress

In [2]:
# ------------------------------------------------------------
# MMASH bridge test: sleep efficiency → stress
# ------------------------------------------------------------

from pathlib import Path
import subprocess
import sys
import zipfile
import warnings

warnings.filterwarnings("ignore")


# ------------------------------------------------------------
# Install packages if needed
# ------------------------------------------------------------

def install_if_missing(package_name, import_name=None):
    if import_name is None:
        import_name = package_name

    try:
        __import__(import_name)
    except ImportError:
        print(f"{package_name} missing. Installing...")
        subprocess.check_call([
            sys.executable,
            "-m",
            "pip",
            "install",
            package_name
        ])


install_if_missing("requests")
install_if_missing("pandas")
install_if_missing("numpy")
install_if_missing("matplotlib")
install_if_missing("scikit-learn", "sklearn")

import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from IPython.display import display

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.dummy import DummyRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error


# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name.lower() == "notebooks":
    PROJECT_DIR = CURRENT_DIR.parent
else:
    PROJECT_DIR = CURRENT_DIR

DATA_DIR = PROJECT_DIR / "data"
GRAPH_DIR = PROJECT_DIR / "Graphs"

MMASH_DIR = DATA_DIR / "mmash"
MMASH_ZIP_PATH = MMASH_DIR / "MMASH.zip"
MMASH_EXTRACT_DIR = MMASH_DIR / "extracted"

for folder in [DATA_DIR, GRAPH_DIR, MMASH_DIR, MMASH_EXTRACT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project folder:")
print(PROJECT_DIR)

print("\nMMASH folder:")
print(MMASH_DIR)


# ------------------------------------------------------------
# Automatically download MMASH from PhysioNet
# ------------------------------------------------------------

mmash_url = "https://physionet.org/files/mmash/1.0.0/MMASH.zip"

if not MMASH_ZIP_PATH.exists():
    print("\nDownloading MMASH dataset from PhysioNet...")
    print(mmash_url)

    response = requests.get(mmash_url, timeout=120)

    if response.status_code != 200:
        raise RuntimeError(
            f"Download failed. Status code: {response.status_code}"
        )

    MMASH_ZIP_PATH.write_bytes(response.content)

    print("Downloaded MMASH ZIP to:")
    print(MMASH_ZIP_PATH)

else:
    print("\nMMASH ZIP already exists:")
    print(MMASH_ZIP_PATH)


# ------------------------------------------------------------
# Extract MMASH
# ------------------------------------------------------------

if len(list(MMASH_EXTRACT_DIR.rglob("*.csv"))) == 0:
    print("\nExtracting MMASH ZIP...")

    with zipfile.ZipFile(MMASH_ZIP_PATH, "r") as zip_ref:
        zip_ref.extractall(MMASH_EXTRACT_DIR)

    print("Extracted to:")
    print(MMASH_EXTRACT_DIR)

else:
    print("\nMMASH already extracted.")


# ------------------------------------------------------------
# Inspect files
# ------------------------------------------------------------

all_csv_files = list(MMASH_EXTRACT_DIR.rglob("*.csv"))

print("\nCSV files found:")
print(len(all_csv_files))

for file_path in all_csv_files[:20]:
    print(file_path.relative_to(MMASH_EXTRACT_DIR))


# ------------------------------------------------------------
# Find participant folders and load sleep/questionnaire/activity
# ------------------------------------------------------------

sleep_files = list(MMASH_EXTRACT_DIR.rglob("sleep.csv"))
questionnaire_files = list(MMASH_EXTRACT_DIR.rglob("questionnaire.csv"))
activity_files = list(MMASH_EXTRACT_DIR.rglob("Activity.csv"))

print("\nSleep files:", len(sleep_files))
print("Questionnaire files:", len(questionnaire_files))
print("Activity files:", len(activity_files))


def participant_id_from_path(file_path):
    """
    MMASH is organized by participant folders.
    This grabs the closest folder name above the file.
    """
    return file_path.parent.name


# ------------------------------------------------------------
# Load sleep data
# ------------------------------------------------------------

sleep_rows = []

for sleep_path in sleep_files:
    participant_id = participant_id_from_path(sleep_path)

    try:
        sleep_df = pd.read_csv(sleep_path)
        sleep_df["participant_id"] = participant_id
        sleep_rows.append(sleep_df)
    except Exception as error:
        print("Could not read sleep file:")
        print(sleep_path)
        print(error)

if len(sleep_rows) == 0:
    raise RuntimeError("No sleep.csv files could be read.")

sleep_all = pd.concat(sleep_rows, ignore_index=True)

print("\nSleep data columns:")
print(list(sleep_all.columns))

display(sleep_all.head())


# ------------------------------------------------------------
# Load questionnaire data
# ------------------------------------------------------------

questionnaire_rows = []

for questionnaire_path in questionnaire_files:
    participant_id = participant_id_from_path(questionnaire_path)

    try:
        questionnaire_df = pd.read_csv(questionnaire_path)
        questionnaire_df["participant_id"] = participant_id
        questionnaire_rows.append(questionnaire_df)
    except Exception as error:
        print("Could not read questionnaire file:")
        print(questionnaire_path)
        print(error)

if len(questionnaire_rows) == 0:
    raise RuntimeError("No questionnaire.csv files could be read.")

questionnaire_all = pd.concat(questionnaire_rows, ignore_index=True)

print("\nQuestionnaire data columns:")
print(list(questionnaire_all.columns))

display(questionnaire_all.head())


# ------------------------------------------------------------
# Load activity data and derive input-like counts/durations
# ------------------------------------------------------------

activity_summary_rows = []

for activity_path in activity_files:
    participant_id = participant_id_from_path(activity_path)

    try:
        activity_df = pd.read_csv(activity_path)

        # Normalize column names lightly
        activity_df.columns = [
            str(column).strip()
            for column in activity_df.columns
        ]

        # Try to identify activity column
        possible_activity_columns = [
            column for column in activity_df.columns
            if "activity" in column.lower()
            or "label" in column.lower()
            or "category" in column.lower()
        ]

        possible_start_columns = [
            column for column in activity_df.columns
            if "start" in column.lower()
        ]

        possible_end_columns = [
            column for column in activity_df.columns
            if "end" in column.lower()
        ]

        summary = {
            "participant_id": participant_id,
            "activity_rows": len(activity_df),
            "screen_events": np.nan,
            "caffeine_events": np.nan,
            "alcohol_events": np.nan,
            "exercise_events": np.nan
        }

        if len(possible_activity_columns) > 0:
            activity_column = possible_activity_columns[0]

            activity_values = pd.to_numeric(
                activity_df[activity_column],
                errors="coerce"
            )

            # MMASH categories from documentation:
            # 4 light movement, 5 medium, 6 heavy,
            # 8 small screen, 9 large screen,
            # 10 caffeine, 12 alcohol
            summary["screen_events"] = int(activity_values.isin([8, 9]).sum())
            summary["caffeine_events"] = int(activity_values.isin([10]).sum())
            summary["alcohol_events"] = int(activity_values.isin([12]).sum())
            summary["exercise_events"] = int(activity_values.isin([4, 5, 6]).sum())

        activity_summary_rows.append(summary)

    except Exception as error:
        print("Could not read activity file:")
        print(activity_path)
        print(error)

activity_summary = pd.DataFrame(activity_summary_rows)

print("\nActivity summary:")
display(activity_summary.head())


# ------------------------------------------------------------
# Build participant-level bridge dataset
# ------------------------------------------------------------

# Because questionnaire.csv appears participant-level,
# we aggregate sleep variables by participant.

sleep_all.columns = [
    str(column).strip()
    for column in sleep_all.columns
]

questionnaire_all.columns = [
    str(column).strip()
    for column in questionnaire_all.columns
]

# Find likely columns
print("\nAvailable sleep columns:")
for column in sleep_all.columns:
    print("-", column)

print("\nAvailable questionnaire columns:")
for column in questionnaire_all.columns:
    print("-", column)


# Manual expected names from PhysioNet documentation
candidate_sleep_efficiency_columns = [
    "Latency Efficiency",
    "Sleep Efficiency",
    "Efficiency"
]

candidate_sleep_quality_columns = [
    "Sleep Fragmentation Index",
    "Fragmentation Index",
    "Movement Index",
    "Total Sleep Time (TST)",
    "WASO"
]

candidate_stress_columns = [
    "Daily_stress",
    "daily_stress",
    "Daily Stress",
    "DSI"
]

candidate_anxiety_columns = [
    "STAI1",
    "STAI2",
    "STAI"
]

candidate_emotion_columns = [
    column for column in questionnaire_all.columns
    if "PANAS" in column.upper()
]


def find_first_existing_column(dataframe, candidates):
    for column in candidates:
        if column in dataframe.columns:
            return column
    return None


sleep_efficiency_column = find_first_existing_column(
    sleep_all,
    candidate_sleep_efficiency_columns
)

stress_column = find_first_existing_column(
    questionnaire_all,
    candidate_stress_columns
)

anxiety_columns = [
    column for column in candidate_anxiety_columns
    if column in questionnaire_all.columns
]

print("\nDetected sleep efficiency column:")
print(sleep_efficiency_column)

print("\nDetected stress column:")
print(stress_column)

print("\nDetected anxiety columns:")
print(anxiety_columns)

print("\nDetected emotion/PANAS columns:")
print(candidate_emotion_columns)


if sleep_efficiency_column is None:
    raise ValueError(
        "Could not find a sleep efficiency column. "
        "Check the printed sleep columns above."
    )

if stress_column is None:
    raise ValueError(
        "Could not find a daily stress column. "
        "Check the printed questionnaire columns above."
    )


# Convert numeric sleep columns
numeric_sleep_columns = [
    sleep_efficiency_column,
    "Total Sleep Time (TST)",
    "WASO",
    "Number of Awakenings",
    "Sleep Fragmentation Index",
    "Movement Index",
    "Fragmentation Index"
]

numeric_sleep_columns = [
    column for column in numeric_sleep_columns
    if column in sleep_all.columns
]

for column in numeric_sleep_columns:
    sleep_all[column] = pd.to_numeric(
        sleep_all[column],
        errors="coerce"
    )

sleep_agg = sleep_all.groupby("participant_id")[numeric_sleep_columns].mean().reset_index()

# Convert numeric questionnaire columns
questionnaire_numeric_candidates = [
    stress_column,
    "STAI1",
    "STAI2",
    "PSQI"
] + candidate_emotion_columns

questionnaire_numeric_candidates = [
    column for column in questionnaire_numeric_candidates
    if column in questionnaire_all.columns
]

for column in questionnaire_numeric_candidates:
    questionnaire_all[column] = pd.to_numeric(
        questionnaire_all[column],
        errors="coerce"
    )

questionnaire_agg = questionnaire_all.groupby("participant_id")[questionnaire_numeric_candidates].mean().reset_index()

mmash_bridge_data = sleep_agg.merge(
    questionnaire_agg,
    on="participant_id",
    how="inner"
)

mmash_bridge_data = mmash_bridge_data.merge(
    activity_summary,
    on="participant_id",
    how="left"
)

print("\nMMASH bridge data:")
display(mmash_bridge_data)

print("\nMMASH bridge data correlations:")
numeric_cols = mmash_bridge_data.select_dtypes(include=[np.number]).columns
display(mmash_bridge_data[numeric_cols].corr().round(3))


# ------------------------------------------------------------
# Helper: evaluate regression path
# ------------------------------------------------------------

def evaluate_path(data, predictors, target, path_name):
    clean_data = data[predictors + [target]].dropna().copy()

    if len(clean_data) < 8:
        return {
            "path": path_name,
            "target": target,
            "predictors": ", ".join(predictors),
            "rows_used": len(clean_data),
            "decision": "not enough rows",
            "raw_formula": "",
            "standardized_formula": ""
        }, pd.DataFrame()

    X = clean_data[predictors]
    y = clean_data[target]

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.30,
        random_state=42
    )

    baseline_model = DummyRegressor(strategy="mean")
    baseline_model.fit(X_train, y_train)
    baseline_predictions = baseline_model.predict(X_test)

    baseline_r2 = r2_score(y_test, baseline_predictions)
    baseline_mae = mean_absolute_error(y_test, baseline_predictions)
    baseline_rmse = np.sqrt(mean_squared_error(y_test, baseline_predictions))

    linear_model = Pipeline(steps=[
        ("scaler", StandardScaler()),
        ("model", LinearRegression())
    ])

    linear_model.fit(X_train, y_train)

    linear_train_predictions = linear_model.predict(X_train)
    linear_test_predictions = linear_model.predict(X_test)

    linear_train_r2 = r2_score(y_train, linear_train_predictions)
    linear_test_r2 = r2_score(y_test, linear_test_predictions)
    linear_mae = mean_absolute_error(y_test, linear_test_predictions)
    linear_rmse = np.sqrt(mean_squared_error(y_test, linear_test_predictions))

    random_forest_model = RandomForestRegressor(
        n_estimators=300,
        max_depth=3,
        min_samples_leaf=2,
        random_state=42
    )

    random_forest_model.fit(X_train, y_train)

    rf_train_predictions = random_forest_model.predict(X_train)
    rf_test_predictions = random_forest_model.predict(X_test)

    rf_train_r2 = r2_score(y_train, rf_train_predictions)
    rf_test_r2 = r2_score(y_test, rf_test_predictions)
    rf_mae = mean_absolute_error(y_test, rf_test_predictions)
    rf_rmse = np.sqrt(mean_squared_error(y_test, rf_test_predictions))

    options = [
        ("baseline mean", baseline_rmse),
        ("linear regression", linear_rmse),
        ("random forest", rf_rmse)
    ]

    best_model, best_rmse = sorted(options, key=lambda item: item[1])[0]
    beats_baseline = best_rmse < baseline_rmse

    decision = (
        "use with caution"
        if beats_baseline and best_model != "baseline mean"
        else "do not use"
    )

    scaler = linear_model.named_steps["scaler"]
    fitted_linear = linear_model.named_steps["model"]

    standardized_intercept = fitted_linear.intercept_
    standardized_coefficients = fitted_linear.coef_

    feature_means = scaler.mean_
    feature_sds = scaler.scale_

    raw_coefficients = standardized_coefficients / feature_sds

    raw_intercept = standardized_intercept - np.sum(
        standardized_coefficients * feature_means / feature_sds
    )

    standardized_parts = [
        f"{standardized_coefficients[i]:+.6f} * z({predictors[i]})"
        for i in range(len(predictors))
    ]

    raw_parts = [
        f"{raw_coefficients[i]:+.6f} * {predictors[i]}"
        for i in range(len(predictors))
    ]

    standardized_formula = (
        f"{target} = {standardized_intercept:.6f} "
        + " ".join(standardized_parts)
    )

    raw_formula = (
        f"{target} = {raw_intercept:.6f} "
        + " ".join(raw_parts)
    )

    result = {
        "path": path_name,
        "target": target,
        "predictors": ", ".join(predictors),
        "rows_used": len(clean_data),
        "baseline_r2": round(baseline_r2, 4),
        "baseline_mae": round(baseline_mae, 4),
        "baseline_rmse": round(baseline_rmse, 4),
        "linear_train_r2": round(linear_train_r2, 4),
        "linear_test_r2": round(linear_test_r2, 4),
        "linear_mae": round(linear_mae, 4),
        "linear_rmse": round(linear_rmse, 4),
        "rf_train_r2": round(rf_train_r2, 4),
        "rf_test_r2": round(rf_test_r2, 4),
        "rf_mae": round(rf_mae, 4),
        "rf_rmse": round(rf_rmse, 4),
        "best_model": best_model,
        "beats_baseline": "Yes" if beats_baseline else "No",
        "decision": decision,
        "raw_formula": raw_formula,
        "standardized_formula": standardized_formula
    }

    coefficient_rows = []

    coefficient_rows.append({
        "path": path_name,
        "term": "Intercept",
        "raw_coefficient": raw_intercept,
        "standardized_coefficient": standardized_intercept,
        "training_mean": "",
        "training_sd": ""
    })

    for i, predictor in enumerate(predictors):
        coefficient_rows.append({
            "path": path_name,
            "term": predictor,
            "raw_coefficient": raw_coefficients[i],
            "standardized_coefficient": standardized_coefficients[i],
            "training_mean": feature_means[i],
            "training_sd": feature_sds[i]
        })

    coefficient_table = pd.DataFrame(coefficient_rows)

    return result, coefficient_table


# ------------------------------------------------------------
# Test MMASH bridge paths
# ------------------------------------------------------------

model_jobs = [
    {
        "path_name": "Sleep efficiency → Daily stress",
        "predictors": [sleep_efficiency_column],
        "target": stress_column
    },
    {
        "path_name": "Sleep efficiency + TST → Daily stress",
        "predictors": [
            column for column in [sleep_efficiency_column, "Total Sleep Time (TST)"]
            if column in mmash_bridge_data.columns
        ],
        "target": stress_column
    },
    {
        "path_name": "Sleep fragmentation → Daily stress",
        "predictors": [
            column for column in ["Sleep Fragmentation Index"]
            if column in mmash_bridge_data.columns
        ],
        "target": stress_column
    },
    {
        "path_name": "Activity inputs → Sleep efficiency",
        "predictors": [
            column for column in ["screen_events", "caffeine_events", "alcohol_events", "exercise_events"]
            if column in mmash_bridge_data.columns
        ],
        "target": sleep_efficiency_column
    }
]

# Add anxiety tests if available
for anxiety_column in anxiety_columns:
    model_jobs.append({
        "path_name": f"Sleep efficiency → {anxiety_column}",
        "predictors": [sleep_efficiency_column],
        "target": anxiety_column
    })

results = []
coefficient_tables = []

for job in model_jobs:
    if len(job["predictors"]) == 0:
        continue

    result, coefficient_table = evaluate_path(
        data=mmash_bridge_data,
        predictors=job["predictors"],
        target=job["target"],
        path_name=job["path_name"]
    )

    results.append(result)

    if len(coefficient_table) > 0:
        coefficient_tables.append(coefficient_table)

mmash_results_table = pd.DataFrame(results)

if len(coefficient_tables) > 0:
    mmash_coefficients_table = pd.concat(coefficient_tables, ignore_index=True)
else:
    mmash_coefficients_table = pd.DataFrame()

print("\nMMASH model results:")
display(mmash_results_table)

print("\nMMASH coefficients:")
display(mmash_coefficients_table)


# ------------------------------------------------------------
# Save results
# ------------------------------------------------------------

results_path = PROJECT_DIR / "mmash_sleep_efficiency_stress_results.csv"
coefficients_path = PROJECT_DIR / "mmash_sleep_efficiency_stress_coefficients.csv"
bridge_data_path = PROJECT_DIR / "mmash_bridge_data.csv"

mmash_results_table.to_csv(results_path, index=False)
mmash_coefficients_table.to_csv(coefficients_path, index=False)
mmash_bridge_data.to_csv(bridge_data_path, index=False)

print("\nSaved MMASH results to:")
print(results_path)

print("\nSaved MMASH coefficients to:")
print(coefficients_path)

print("\nSaved MMASH bridge data to:")
print(bridge_data_path)


# ------------------------------------------------------------
# Print important formulas clearly
# ------------------------------------------------------------

print("\nIMPORTANT MMASH FORMULAS")
print("=" * 80)

for _, row in mmash_results_table.iterrows():
    print("\n" + row["path"])
    print("-" * 80)
    print(f"Rows used: {row.get('rows_used', '')}")
    print(f"Baseline RMSE: {row.get('baseline_rmse', '')}")
    print(f"Linear test R²: {row.get('linear_test_r2', '')}")
    print(f"Linear RMSE: {row.get('linear_rmse', '')}")
    print(f"Random forest test R²: {row.get('rf_test_r2', '')}")
    print(f"Random forest RMSE: {row.get('rf_rmse', '')}")
    print(f"Best model: {row.get('best_model', '')}")
    print(f"Beats baseline: {row.get('beats_baseline', '')}")
    print(f"Decision: {row.get('decision', '')}")

    print("\nRaw formula:")
    print(row.get("raw_formula", ""))

    print("\nStandardized formula:")
    print(row.get("standardized_formula", ""))


# ------------------------------------------------------------
# Plot RMSE comparison
# ------------------------------------------------------------

plot_data = mmash_results_table[
    mmash_results_table["decision"] != "not enough rows"
].copy()

if len(plot_data) > 0 and "linear_rmse" in plot_data.columns:
    plt.figure(figsize=(10, 6))
    plt.barh(plot_data["path"], plot_data["linear_rmse"])
    plt.xlabel("Linear model test RMSE")
    plt.ylabel("Path")
    plt.title("MMASH sleep-efficiency bridge comparison")
    plt.tight_layout()

    graph_path = GRAPH_DIR / "mmash_sleep_efficiency_bridge_rmse.png"
    plt.savefig(graph_path, dpi=200)
    plt.close()

    print("\nSaved graph to:")
    print(graph_path)

Project folder:
D:\Documentos\Sleep datasets\sleep_installation_data_analysis

MMASH folder:
D:\Documentos\Sleep datasets\sleep_installation_data_analysis\data\mmash

MMASH ZIP already exists:
D:\Documentos\Sleep datasets\sleep_installation_data_analysis\data\mmash\MMASH.zip

MMASH already extracted.

CSV files found:
153
DataPaper\user_1\Actigraph.csv
DataPaper\user_1\Activity.csv
DataPaper\user_1\questionnaire.csv
DataPaper\user_1\RR.csv
DataPaper\user_1\saliva.csv
DataPaper\user_1\sleep.csv
DataPaper\user_1\user_info.csv
DataPaper\user_10\Actigraph.csv
DataPaper\user_10\Activity.csv
DataPaper\user_10\questionnaire.csv
DataPaper\user_10\RR.csv
DataPaper\user_10\saliva.csv
DataPaper\user_10\sleep.csv
DataPaper\user_10\user_info.csv
DataPaper\user_11\Actigraph.csv
DataPaper\user_11\Activity.csv
DataPaper\user_11\questionnaire.csv
DataPaper\user_11\RR.csv
DataPaper\user_11\saliva.csv
DataPaper\user_11\sleep.csv

Sleep files: 22
Questionnaire files: 22
Activity files: 22

Sleep data colu

,Unnamed: 0,In Bed Date,In Bed Time,Out Bed Date,Out Bed Time,Onset Date,Onset Time,Latency,Efficiency,Total Minutes in Bed,Total Sleep Time (TST),Wake After Sleep Onset (WASO),Number of Awakenings,Average Awakening Length,Movement Index,Fragmentation Index,Sleep Fragmentation Index,participant_id
0,0,2,00:46,2,03:31,2,00:46,0,87.27,165,144,21,9,2.33,9.091,10,19.091,user_1
1,1,2,03:57,2,07:30,2,03:57,0,92.02,213,196,17,9,1.89,8.92,0,8.92,user_1
2,0,2,02:40,2,08:09,2,02:44,4,75.08,329,247,78,13,6,20.669,7.692,28.361,user_10
3,0,2,00:32,2,06:36,2,00:32,0,94.23,364,343,21,12,1.75,9.066,15.385,24.451,user_12
4,0,1,23:41,1,05:04,1,23:41,0,76.47,323,247,76,19,4,17.957,15.789,33.746,user_13



Questionnaire data columns:
['Unnamed: 0', 'MEQ', 'STAI1', 'STAI2', 'Pittsburgh', 'Daily_stress', 'BISBAS_bis', 'BISBAS_reward', 'BISBAS_drive', 'BISBAS_fun', 'panas_pos_10', 'panas_pos_14', 'panas_pos_18', 'panas_pos_22', 'panas_pos_9+1', 'panas_neg_10', 'panas_neg_14', 'panas_neg_18', 'panas_neg_22', 'panas_neg_9+1', 'participant_id']


,Unnamed: 0,MEQ,STAI1,STAI2,Pittsburgh,Daily_stress,BISBAS_bis,BISBAS_reward,BISBAS_drive,BISBAS_fun,...,panas_pos_14,panas_pos_18,panas_pos_22,panas_pos_9+1,panas_neg_10,panas_neg_14,panas_neg_18,panas_neg_22,panas_neg_9+1,participant_id
0,0,47.0,41.0,43.0,5.0,23.0,22.0,21.0,14.0,14.0,...,17.0,12.0,18.0,17.0,11.0,13.0,13.0,10.0,10.0,user_1
1,0,38.0,39.0,46.0,4.0,14.0,19.0,16.0,16.0,14.0,...,23.0,21.0,18.0,23.0,15.0,23.0,23.0,27.0,12.0,user_10
2,0,38.0,36.0,43.0,7.0,17.0,23.0,19.0,11.0,13.0,...,16.0,22.0,30.0,13.0,15.0,15.0,11.0,13.0,14.0,user_11
3,0,50.0,27.0,33.0,4.0,48.0,20.0,22.0,10.0,8.0,...,27.0,30.0,20.0,29.0,14.0,16.0,14.0,23.0,13.0,user_12
4,0,48.0,30.0,43.0,4.0,27.0,22.0,19.0,14.0,12.0,...,NaN,26.0,14.0,23.0,13.0,NaN,11.0,13.0,12.0,user_13



Activity summary:


...


MMASH model results:


,path,target,predictors,rows_used,baseline_r2,baseline_mae,baseline_rmse,linear_train_r2,linear_test_r2,linear_mae,linear_rmse,rf_train_r2,rf_test_r2,rf_mae,rf_rmse,best_model,beats_baseline,decision,raw_formula,standardized_formula
0,Sleep efficiency → Daily stress,Daily_stress,Efficiency,21,-1.1976,12.6531,15.4814,0.0003,-1.1904,12.5802,15.4559,0.3222,-1.7677,14.0584,17.3738,linear regression,Yes,use with caution,Daily_stress = 39.844312 -0.037778 * Efficiency,Daily_stress = 36.714286 -0.273687 * z(Efficie...
1,Sleep efficiency + TST → Daily stress,Daily_stress,"Efficiency, Total Sleep Time (TST)",21,-1.1976,12.6531,15.4814,0.0239,-0.8439,11.3166,14.1808,0.5175,-0.8710,11.7298,14.2848,linear regression,Yes,use with caution,Daily_stress = 40.194577 -0.191276 * Efficienc...,Daily_stress = 36.714286 -1.385721 * z(Efficie...
2,Sleep fragmentation → Daily stress,Daily_stress,Sleep Fragmentation Index,21,-1.1976,12.6531,15.4814,0.2874,-1.0474,12.2849,14.9429,0.6533,-1.0335,12.4342,14.8921,random forest,Yes,use with caution,Daily_stress = 17.223861 +0.750613 * Sleep Fra...,Daily_stress = 36.714286 +9.019695 * z(Sleep F...
3,Activity inputs → Sleep efficiency,Efficiency,"screen_events, caffeine_events, alcohol_events...",21,-0.2567,4.5630,5.1821,0.3083,-3.3724,8.2165,9.6661,0.3860,-1.1180,5.9368,6.7275,baseline mean,No,do not use,Efficiency = 73.006801 -1.311610 * screen_even...,Efficiency = 82.852857 -1.910836 * z(screen_ev...
4,Sleep efficiency → STAI1,STAI1,Efficiency,21,-0.3095,6.9694,8.6688,0.2541,-0.0205,6.5426,7.6527,0.6577,-0.7632,8.6828,10.0591,linear regression,Yes,use with caution,STAI1 = -13.826817 +0.579836 * Efficiency,STAI1 = 34.214286 +4.200681 * z(Efficiency)
5,Sleep efficiency → STAI2,STAI2,Efficiency,21,-7.3960,6.1429,6.5450,0.0946,-7.1048,4.9555,6.4305,0.4501,-10.2234,4.6635,7.5672,linear regression,Yes,use with caution,STAI2 = -4.716825 +0.506953 * Efficiency,STAI2 = 37.285714 +3.672673 * z(Efficiency)



MMASH coefficients:


,path,term,raw_coefficient,standardized_coefficient,training_mean,training_sd
0,Sleep efficiency → Daily stress,Intercept,3.984431e+01,3.671429e+01,,
1,Sleep efficiency → Daily stress,Efficiency,-3.777814e-02,-2.736874e-01,82.852857,7.244597
2,Sleep efficiency + TST → Daily stress,Intercept,4.019458e+01,3.671429e+01,,
3,Sleep efficiency + TST → Daily stress,Efficiency,-1.912765e-01,-1.385721e+00,82.852857,7.244597
4,Sleep efficiency + TST → Daily stress,Total Sleep Time (TST),3.664448e-02,2.816990e+00,337.5,76.873505
5,Sleep fragmentation → Daily stress,Intercept,1.722386e+01,3.671429e+01,,
6,Sleep fragmentation → Daily stress,Sleep Fragmentation Index,7.506133e-01,9.019695e+00,25.966,12.016434
7,Activity inputs → Sleep efficiency,Intercept,7.300680e+01,8.285286e+01,,
8,Activity inputs → Sleep efficiency,screen_events,-1.311610e+00,-1.910836e+00,2.142857,1.456863
9,Activity inputs → Sleep efficiency,caffeine_events,-8.881784e-16,-8.881784e-16,0.0,1.0



Saved MMASH results to:
D:\Documentos\Sleep datasets\sleep_installation_data_analysis\mmash_sleep_efficiency_stress_results.csv

Saved MMASH coefficients to:
D:\Documentos\Sleep datasets\sleep_installation_data_analysis\mmash_sleep_efficiency_stress_coefficients.csv

Saved MMASH bridge data to:
D:\Documentos\Sleep datasets\sleep_installation_data_analysis\mmash_bridge_data.csv

IMPORTANT MMASH FORMULAS

Sleep efficiency → Daily stress
--------------------------------------------------------------------------------
Rows used: 21
Baseline RMSE: 15.4814
Linear test R²: -1.1904
Linear RMSE: 15.4559
Random forest test R²: -1.7677
Random forest RMSE: 17.3738
Best model: linear regression
Beats baseline: Yes
Decision: use with caution

Raw formula:
Daily_stress = 39.844312 -0.037778 * Efficiency

Standardized formula:
Daily_stress = 36.714286 -0.273687 * z(Efficiency)

Sleep efficiency + TST → Daily stress
--------------------------------------------------------------------------------
Rows 

In [3]:
# ------------------------------------------------------------
# Stricter MMASH decision table
# ------------------------------------------------------------

strict_mmash_table = mmash_results_table.copy()

def strict_decision(row):
    if row.get("decision") == "not enough rows":
        return "not enough rows"

    baseline_rmse = row.get("baseline_rmse")
    linear_rmse = row.get("linear_rmse")
    linear_test_r2 = row.get("linear_test_r2")

    if pd.isna(baseline_rmse) or pd.isna(linear_rmse) or pd.isna(linear_test_r2):
        return "do not use"

    rmse_improvement = baseline_rmse - linear_rmse
    rmse_improvement_percent = rmse_improvement / baseline_rmse

    if linear_test_r2 > 0 and rmse_improvement_percent >= 0.05:
        return "use with caution"

    if rmse_improvement > 0:
        return "too weak: tiny RMSE improvement or negative test R²"

    return "do not use"

strict_mmash_table["rmse_improvement"] = (
    strict_mmash_table["baseline_rmse"] - strict_mmash_table["linear_rmse"]
)

strict_mmash_table["rmse_improvement_percent"] = (
    strict_mmash_table["rmse_improvement"] / strict_mmash_table["baseline_rmse"]
)

strict_mmash_table["strict_decision"] = strict_mmash_table.apply(
    strict_decision,
    axis=1
)

display(strict_mmash_table[
    [
        "path",
        "rows_used",
        "baseline_rmse",
        "linear_test_r2",
        "linear_rmse",
        "rmse_improvement",
        "rmse_improvement_percent",
        "decision",
        "strict_decision",
        "raw_formula"
    ]
])

,path,rows_used,baseline_rmse,linear_test_r2,linear_rmse,rmse_improvement,rmse_improvement_percent,decision,strict_decision,raw_formula
0,Sleep efficiency → Daily stress,21,15.4814,-1.1904,15.4559,0.0255,0.001647,use with caution,too weak: tiny RMSE improvement or negative te...,Daily_stress = 39.844312 -0.037778 * Efficiency
1,Sleep efficiency + TST → Daily stress,21,15.4814,-0.8439,14.1808,1.3006,0.084010,use with caution,too weak: tiny RMSE improvement or negative te...,Daily_stress = 40.194577 -0.191276 * Efficienc...
2,Sleep fragmentation → Daily stress,21,15.4814,-1.0474,14.9429,0.5385,0.034784,use with caution,too weak: tiny RMSE improvement or negative te...,Daily_stress = 17.223861 +0.750613 * Sleep Fra...
3,Activity inputs → Sleep efficiency,21,5.1821,-3.3724,9.6661,-4.4840,-0.865286,do not use,do not use,Efficiency = 73.006801 -1.311610 * screen_even...
4,Sleep efficiency → STAI1,21,8.6688,-0.0205,7.6527,1.0161,0.117213,use with caution,too weak: tiny RMSE improvement or negative te...,STAI1 = -13.826817 +0.579836 * Efficiency
5,Sleep efficiency → STAI2,21,6.5450,-7.1048,6.4305,0.1145,0.017494,use with caution,too weak: tiny RMSE improvement or negative te...,STAI2 = -4.716825 +0.506953 * Efficiency


## MMASH Mood / Emotion Bridge Test

The previous MMASH stress models were too weak to rely on.

This step checks whether MMASH can still help with the mood/emotion output.

Target paths:

sleep efficiency → PANAS positive affect
sleep efficiency → PANAS negative affect

If these models beat baseline with a meaningful improvement, MMASH can support a mood/emotion output path.

# ------------------------------------------------------------
# MMASH mood / emotion bridge test
# ------------------------------------------------------------

import pandas as pd
import numpy as np

from IPython.display import display

# ------------------------------------------------------------
# Safety checks
# ------------------------------------------------------------

required_objects = [
    "mmash_bridge_data",
    "sleep_efficiency_column",
    "evaluate_path",
    "PROJECT_DIR"
]

missing_objects = [
    name for name in required_objects
    if name not in globals()
]

if len(missing_objects) > 0:
    print("Missing required objects:")
    for name in missing_objects:
        print(f"- {name}")

    print("\nPlease run the earlier MMASH cells first.")

else:
    print("Available columns in MMASH bridge data:")
    for column in mmash_bridge_data.columns:
        print("-", column)

    # --------------------------------------------------------
    # Find possible mood/emotion columns
    # --------------------------------------------------------

    mood_keywords = [
        "PANAS",
        "Positive",
        "Negative",
        "Affect",
        "Mood",
        "Emotion",
        "Happy",
        "Sad",
        "Angry",
        "Calm",
        "Fatigue",
        "Tired"
    ]

    possible_mood_columns = []

    for column in mmash_bridge_data.columns:
        column_lower = str(column).lower()

        if any(keyword.lower() in column_lower for keyword in mood_keywords):
            possible_mood_columns.append(column)

    print("\nPossible mood/emotion columns:")
    for column in possible_mood_columns:
        print("-", column)

    # --------------------------------------------------------
    # Build model jobs
    # --------------------------------------------------------

    mood_model_jobs = []

    for mood_column in possible_mood_columns:
        if mood_column == sleep_efficiency_column:
            continue

        mood_model_jobs.append({
            "path_name": f"Sleep efficiency → {mood_column}",
            "predictors": [sleep_efficiency_column],
            "target": mood_column
        })

    if len(mood_model_jobs) == 0:
        print("\nNo mood/emotion columns found.")
        print("Send me the available columns list so we can inspect manually.")

    else:
        mood_results = []
        mood_coefficient_tables = []

        for job in mood_model_jobs:
            result, coefficient_table = evaluate_path(
                data=mmash_bridge_data,
                predictors=job["predictors"],
                target=job["target"],
                path_name=job["path_name"]
            )

            mood_results.append(result)

            if len(coefficient_table) > 0:
                mood_coefficient_tables.append(coefficient_table)

        mmash_mood_results_table = pd.DataFrame(mood_results)

        if len(mood_coefficient_tables) > 0:
            mmash_mood_coefficients_table = pd.concat(
                mood_coefficient_tables,
                ignore_index=True
            )
        else:
            mmash_mood_coefficients_table = pd.DataFrame()

        # ----------------------------------------------------
        # Strict decision rule
        # ----------------------------------------------------

        def strict_decision(row):
            if row.get("decision") == "not enough rows":
                return "not enough rows"

            baseline_rmse = row.get("baseline_rmse")
            linear_rmse = row.get("linear_rmse")
            linear_test_r2 = row.get("linear_test_r2")

            if pd.isna(baseline_rmse) or pd.isna(linear_rmse) or pd.isna(linear_test_r2):
                return "do not use"

            rmse_improvement = baseline_rmse - linear_rmse
            rmse_improvement_percent = rmse_improvement / baseline_rmse

            if linear_test_r2 > 0 and rmse_improvement_percent >= 0.05:
                return "use with caution"

            if rmse_improvement > 0:
                return "too weak: tiny RMSE improvement or negative test R²"

            return "do not use"

        if "baseline_rmse" in mmash_mood_results_table.columns:
            mmash_mood_results_table["rmse_improvement"] = (
                mmash_mood_results_table["baseline_rmse"]
                - mmash_mood_results_table["linear_rmse"]
            )

            mmash_mood_results_table["rmse_improvement_percent"] = (
                mmash_mood_results_table["rmse_improvement"]
                / mmash_mood_results_table["baseline_rmse"]
            )

            mmash_mood_results_table["strict_decision"] = (
                mmash_mood_results_table.apply(strict_decision, axis=1)
            )

        print("\nMMASH mood/emotion model results:")
        display(mmash_mood_results_table)

        print("\nMMASH mood/emotion coefficients:")
        display(mmash_mood_coefficients_table)

        # ----------------------------------------------------
        # Save results
        # ----------------------------------------------------

        mood_results_path = PROJECT_DIR / "mmash_sleep_efficiency_mood_results.csv"
        mood_coefficients_path = PROJECT_DIR / "mmash_sleep_efficiency_mood_coefficients.csv"

        mmash_mood_results_table.to_csv(mood_results_path, index=False)
        mmash_mood_coefficients_table.to_csv(mood_coefficients_path, index=False)

        print("\nSaved MMASH mood results to:")
        print(mood_results_path)

        print("\nSaved MMASH mood coefficients to:")
        print(mood_coefficients_path)

        # ----------------------------------------------------
        # Print formulas clearly
        # ----------------------------------------------------

        print("\nIMPORTANT MMASH MOOD FORMULAS")
        print("=" * 80)

        for _, row in mmash_mood_results_table.iterrows():
            print("\n" + row["path"])
            print("-" * 80)
            print(f"Rows used: {row.get('rows_used', '')}")
            print(f"Baseline RMSE: {row.get('baseline_rmse', '')}")
            print(f"Linear test R²: {row.get('linear_test_r2', '')}")
            print(f"Linear RMSE: {row.get('linear_rmse', '')}")
            print(f"RMSE improvement percent: {row.get('rmse_improvement_percent', '')}")
            print(f"Original decision: {row.get('decision', '')}")
            print(f"Strict decision: {row.get('strict_decision', '')}")

            print("\nRaw formula:")
            print(row.get("raw_formula", ""))

            print("\nStandardized formula:")
            print(row.get("standardized_formula", ""))